# Policy Engine & Validation Agent — LangGraph Flow Scaffold

This notebook demonstrates how to use and compose the transaction validation policy engine (`app.agent`) inside a **LangGraph** workflow.

### Architecture Summary (see `docs/contracts.md`)
- **Public service entry points** (`app.agent.service`): `run_evaluate(...)` and `run_review(...)`.
- **Compiled LangGraph subgraphs** (`app.agent.graphs`): `evaluate_graph` and `review_graph`.
- **Invariants**:
  - Hard policy checks (`app.agent.checks.CHECKS`) are deterministic Python.
  - Overall `recommendation` (`pass` / `no_pass`) is computed in Python, never by the LLM.
  - Every database write goes through `write_with_audit` (hash-chained `audit_log`).
  - `check_gate` enforces role, amount limits, and hard-fail override rules before any decision is recorded.

## 1. Environment & Path Setup

Run this notebook using the `backend/.venv` Python kernel (e.g. `cd backend && uv run jupyter lab` or select `backend/.venv` in VS Code / Cursor).  
This cell adds `backend/` to `sys.path` and loads `.env`.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Resolve project root whether launched from root or backend/
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "backend").is_dir() else CWD.parent
BACKEND_DIR = PROJECT_ROOT / "backend"

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

load_dotenv(PROJECT_ROOT / ".env")
load_dotenv(BACKEND_DIR / ".env")

print(f"Project root : {PROJECT_ROOT}")
print(f"Backend dir  : {BACKEND_DIR}")
print(f"DB_NAME      : {os.getenv('DB_NAME', 'txn_validation')}")
print(f"LLM_MODEL    : {os.getenv('LLM_MODEL', 'google_genai:gemini-2.5-flash')}")

In [ ]:
# Public entry points
from app.agent import run_evaluate, run_review

# Underlying compiled LangGraph subgraphs & state definitions
from app.agent.graphs import (
    EvalState,
    ReviewState,
    evaluate_graph,
    review_graph,
)

print("Loaded evaluate_graph:", type(evaluate_graph))
print("Loaded review_graph  :", type(review_graph))

## 2. Inspecting the Built-in Subgraphs

Both `evaluate_graph` and `review_graph` are standard compiled `StateGraph` instances. You can inspect their Mermaid topology directly.

In [ ]:
print("=== Evaluate Graph (Mermaid) ===")
print(evaluate_graph.get_graph().draw_mermaid())

print("=== Review Graph (Mermaid) ===")
print(review_graph.get_graph().draw_mermaid())

## 3. Direct Invocation via `run_evaluate` and `run_review`

For most application code (like FastAPI routes), call the high-level async functions:
- `await run_evaluate(txn_id="TXN-0001", principal_id="ana")`
- `await run_review(eval_id="EVAL-...", principal_id="ana", review_input={"kind": "question", "question": "..."})`
- `await run_review(eval_id="EVAL-...", principal_id="ana", review_input={"kind": "decision", "verdict": "validated", "comments": None})`

In [ ]:
async def demo_direct_flow(txn_id: str = "TXN-0001", principal_id: str = "ana") -> dict:
    """Run evaluation -> optional Q&A -> decision using the public service interface."""
    # 1. Evaluate transaction against active policy
    eval_res = await run_evaluate(txn_id=txn_id, principal_id=principal_id)
    evaluation = eval_res["evaluation"]
    eval_id = evaluation["eval_id"]
    print(f"Evaluation {eval_id}: recommendation={evaluation['recommendation']}")

    # 2. Ask a reviewer question (writes `question.answered` to audit_log)
    qa_res = await run_review(
        eval_id=eval_id,
        principal_id=principal_id,
        review_input={
            "kind": "question",
            "question": "Which policy criteria failed or flagged, and why?",
        },
    )
    print("Answer:", qa_res["answer"])

    # 3. Submit a decision (runs policy_gate before recording)
    decision_res = await run_review(
        eval_id=eval_id,
        principal_id=principal_id,
        review_input={
            "kind": "decision",
            "verdict": "validated_with_comments",
            "comments": "Reviewed in notebook scaffold; supporting context verified.",
        },
    )
    print("Gate result:", decision_res["gate"])
    return decision_res

## 4. Composing the Policy Agent in a Custom Parent LangGraph Flow

When building a larger orchestration pipeline (e.g., batch triage, automated escalation on `gate_denied`, or multi-step human-in-the-loop review), you can wrap `evaluate_graph` and `review_graph` inside a parent `StateGraph`.

Below is a complete scaffold showing:
1. **`intake`** — normalizes the incoming transaction ID and acting principal.
2. **`evaluate`** — invokes `evaluate_graph`.
3. **`route_after_eval`** — branches based on `recommendation` (`pass` vs `no_pass`) or policy flags.
4. **`triage_question`** — invokes `review_graph` in `question` mode to summarize risks for the reviewer.
5. **`decide`** — invokes `review_graph` in `decision` mode.
6. **`escalate_to_supervisor`** — handles `gate["result"] == "deny"` (e.g., amount exceeds analyst limit) by routing to a supervisor principal.

In [ ]:
from typing import Any, Dict, List, Literal, Optional, TypedDict
from langgraph.graph import END, START, StateGraph


class OrchestratorState(TypedDict, total=False):
    """Parent LangGraph state wrapping the Policy & Validation subgraphs."""

    txn_id: str
    principal_id: str
    supervisor_id: str
    auto_approve_if_pass: bool

    # Outputs populated by subgraphs
    eval_id: str
    evaluation: Dict[str, Any]
    triage_answer: Dict[str, Any]
    gate: Dict[str, Any]
    decision: Optional[Dict[str, Any]]
    memory_created: List[Dict[str, Any]]
    outcome_status: str


async def intake_node(state: OrchestratorState) -> Dict[str, Any]:
    """Prepare defaults for the orchestration run."""
    return {
        "supervisor_id": state.get("supervisor_id", "sam"),
        "auto_approve_if_pass": state.get("auto_approve_if_pass", False),
    }


async def evaluate_node(state: OrchestratorState) -> Dict[str, Any]:
    """Invoke the Policy Engine `evaluate_graph` subgraph."""
    eval_state: EvalState = await evaluate_graph.ainvoke(
        {
            "txn_id": state["txn_id"],
            "principal_id": state["principal_id"],
        }
    )
    evaluation = eval_state["evaluation"]
    return {
        "eval_id": evaluation["eval_id"],
        "evaluation": evaluation,
    }


def route_after_eval(state: OrchestratorState) -> Literal["auto_decide", "triage_question"]:
    """Route clean `pass` recommendations to auto-decide if enabled; otherwise triage."""
    rec = state["evaluation"]["recommendation"]
    if rec == "pass" and state.get("auto_approve_if_pass", False):
        return "auto_decide"
    return "triage_question"


async def triage_question_node(state: OrchestratorState) -> Dict[str, Any]:
    """Ask the agent to summarize failed/flagged criteria before human decision."""
    review_state: ReviewState = await review_graph.ainvoke(
        {
            "eval_id": state["eval_id"],
            "principal_id": state["principal_id"],
            "input": {
                "kind": "question",
                "question": "Summarize the key findings, precedents, and any active adjustments.",
            },
        }
    )
    return {"triage_answer": review_state.get("answer", {})}


async def decide_node(state: OrchestratorState) -> Dict[str, Any]:
    """Attempt to record a decision as the current `principal_id`."""
    rec = state["evaluation"]["recommendation"]
    verdict = "validated" if rec == "pass" else "rejected"
    comments = None if verdict == "validated" else "Rejected based on policy evaluation."

    review_state: ReviewState = await review_graph.ainvoke(
        {
            "eval_id": state["eval_id"],
            "principal_id": state["principal_id"],
            "input": {
                "kind": "decision",
                "verdict": verdict,
                "comments": comments,
            },
        }
    )
    gate = review_state.get("gate", {})
    return {
        "gate": gate,
        "decision": review_state.get("decision"),
        "memory_created": review_state.get("memory_created", []),
        "outcome_status": "decided" if gate.get("result") == "allow" else "gate_denied",
    }


def route_after_decide(state: OrchestratorState) -> Literal["escalate_to_supervisor", "done"]:
    """If the policy gate denied the analyst, escalate to a supervisor."""
    if state.get("gate", {}).get("result") == "deny":
        return "escalate_to_supervisor"
    return "done"


async def escalate_to_supervisor_node(state: OrchestratorState) -> Dict[str, Any]:
    """Placeholder for supervisor escalation (e.g., re-invoking review_graph as supervisor)."""
    return {
        "outcome_status": f"escalated_to:{state['supervisor_id']} (reason: {state['gate'].get('reason')})"
    }


def build_parent_flow():
    graph = StateGraph(OrchestratorState)
    graph.add_node("intake", intake_node)
    graph.add_node("evaluate", evaluate_node)
    graph.add_node("triage_question", triage_question_node)
    graph.add_node("auto_decide", decide_node)
    graph.add_node("decide", decide_node)
    graph.add_node("escalate_to_supervisor", escalate_to_supervisor_node)

    graph.add_edge(START, "intake")
    graph.add_edge("intake", "evaluate")
    graph.add_conditional_edges(
        "evaluate",
        route_after_eval,
        {"auto_decide": "auto_decide", "triage_question": "triage_question"},
    )
    graph.add_edge("triage_question", "decide")
    graph.add_conditional_edges(
        "auto_decide",
        route_after_decide,
        {"escalate_to_supervisor": "escalate_to_supervisor", "done": END},
    )
    graph.add_conditional_edges(
        "decide",
        route_after_decide,
        {"escalate_to_supervisor": "escalate_to_supervisor", "done": END},
    )
    graph.add_edge("escalate_to_supervisor", END)
    return graph.compile()


parent_flow = build_parent_flow()
print("=== Parent Orchestrator Flow (Mermaid) ===")
print(parent_flow.get_graph().draw_mermaid())

## 5. Run the Scaffold

Set `RUN_LIVE = True` once your `.env` is configured and MongoDB is seeded (`cd backend && uv run python -m scripts.seed`).

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    result = await parent_flow.ainvoke(
        {
            "txn_id": "TXN-0001",
            "principal_id": "ana",
            "supervisor_id": "sam",
            "auto_approve_if_pass": True,
        }
    )
    print("Outcome status:", result.get("outcome_status"))
    print("Eval ID       :", result.get("eval_id"))
    print("Gate          :", result.get("gate"))
else:
    print("Dry run: parent_flow compiled successfully. Set RUN_LIVE = True to execute against MongoDB + LLM.")